<a href="https://colab.research.google.com/github/FriendsofHilma/hear-ontology/blob/main/dissertation_to_opencode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dissertation → OpenCode research package

This notebook prepares a PDF dissertation in Google Drive for use as a research source in an OpenCode project.

**What it creates**
- `dissertation.md` — page-labelled text, with figure/image references
- `pages.jsonl` — one record per PDF page (including page number and extracted text)
- `chunks.jsonl` — overlapping text chunks with source-page metadata
- `figures/` — embedded images extracted from the PDF
- `page_renders/` — optional page images for scanned or visually important pages
- `docling.md` — optional structured Markdown export from Docling
- `manifest.json` — processing details
- a ZIP archive ready to copy into your OpenCode project

The original PDF is never modified. The notebook works on a copy/read-only input.

**How to use:** Put the PDF in a folder in Google Drive shared with the collaborators who need access, edit the PDF path in the setup cell, then run the notebook from top to bottom. Colab may ask you to authorize Drive access.

## 1. Install the tools

PyMuPDF provides reliable page-level text and embedded-image extraction. Tesseract OCR is used only for pages with little or no extractable text. Docling is included as an additional layout-aware document conversion pass; if it fails or takes too many resources, the rest of the pipeline can still complete.

In [10]:

!apt-get -qq install -y tesseract-ocr tesseract-ocr-swe


Selecting previously unselected package tesseract-ocr-swe.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-swe_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-swe (1:4.1.0-2) ...
Setting up tesseract-ocr-swe (1:4.1.0-2) ...


## 2. Connect Google Drive and choose the PDF

Change `PDF_PATH` to the actual location of the dissertation in Drive. The output folder will be created beside it unless you change `OUTPUT_DIR`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# EDIT THIS: use the path to the PDF in your Google Drive.
PDF_PATH = Path("/content/drive/MyDrive/Hilma Research/Rachels dissertation/HafK_Atom-Tree_Text_PROOF5_6-25-26.pdf")

# Optional: set a different output folder. None = create "dissertation_research_package"
# next to the PDF.
OUTPUT_DIR = None

# Processing options
RUN_DOCLING = True           # Set False if you want a faster/lighter first run.
RUN_OCR_FOR_IMAGE_PAGES = True
RENDER_LOW_TEXT_PAGES = True
CHUNK_WORDS = 800
CHUNK_OVERLAP_WORDS = 120
IMAGE_DPI = 150              # Higher DPI gives larger page images.

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH}\n"
        "Edit PDF_PATH above so it points to the PDF in your Google Drive."
    )

if OUTPUT_DIR is None:
    OUTPUT_DIR = PDF_PATH.parent / "dissertation_research_package"
else:
    OUTPUT_DIR = Path(OUTPUT_DIR)

FIGURES_DIR = OUTPUT_DIR / "figures"
PAGE_RENDERS_DIR = OUTPUT_DIR / "page_renders"
for folder in (OUTPUT_DIR, FIGURES_DIR, PAGE_RENDERS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Input:  {PDF_PATH}")
print(f"Output: {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input:  /content/drive/MyDrive/Hilma Research/Rachels dissertation/HafK_Atom-Tree_Text_PROOF5_6-25-26.pdf
Output: /content/drive/MyDrive/Hilma Research/Rachels dissertation/dissertation_research_package


In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.lower().endswith('.pdf'):
            print(os.path.join(root, file))

/content/drive/MyDrive/Hilma Research/Rachels dissertation/HafK_Atom-Tree_Text_PROOF5_6-25-26.pdf


import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.lower().endswith('.pdf'):
            print(os.path.join(root, file))

## 3. Extract page text, embedded images, and OCR where needed

Every page keeps its original PDF page number (starting at 1). Pages with little extractable text can be OCR-processed. Embedded images are extracted separately and linked back to the page where they appear.

Note: some PDF figures are vector drawings rather than embedded raster images. Those remain visible in the original PDF and in the optional page renders; they may not appear as separate files in `figures/`.

In [ ]:
import fitz  # PyMuPDF
from PIL import Image
import pytesseract
import hashlib, json, re, io, os, sys, traceback
from datetime import datetime, timezone

pdf = fitz.open(PDF_PATH)
page_records = []
image_records = []
markdown_parts = []
seen_images = set()

def safe_stem(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "-", s).strip("-")[:80] or "image"

for page_index, page in enumerate(pdf):
    page_num = page_index + 1
    text = page.get_text("text", sort=True).strip()
    ocr_used = False

    # A practical heuristic: OCR pages with very little selectable text.
    if RUN_OCR_FOR_IMAGE_PAGES and len(text) < 80:
        try:
            pix = page.get_pixmap(dpi=IMAGE_DPI, alpha=False)
            pil_img = Image.open(io.BytesIO(pix.tobytes("png")))
            ocr_text = pytesseract.image_to_string(pil_img).strip()
            if len(ocr_text) > len(text):
                text = ocr_text
                ocr_used = True
        except Exception as e:
            print(f"OCR warning on page {page_num}: {e}")

    page_image_names = []
    for img_no, img in enumerate(page.get_images(full=True), start=1):
        xref = img[0]
        try:
            extracted = pdf.extract_image(xref)
            image_bytes = extracted["image"]
            digest = hashlib.sha256(image_bytes).hexdigest()[:16]
            if digest in seen_images:
                continue
            seen_images.add(digest)
            ext = extracted.get("ext", "bin")
            filename = f"page-{page_num:04d}-image-{img_no:02d}-{digest}.{ext}"
            out_path = FIGURES_DIR / filename
            out_path.write_bytes(image_bytes)
            page_image_names.append(filename)
            image_records.append({
                "file": f"figures/{filename}",
                "page": page_num,
                "width": extracted.get("width"),
                "height": extracted.get("height"),
                "extension": ext,
                "sha256_prefix": digest
            })
        except Exception as e:
            print(f"Image extraction warning on page {page_num}, image {img_no}: {e}")

    # Save a page render when the page is likely image-based or has embedded images.
    render_name = None
    if RENDER_LOW_TEXT_PAGES and (len(text) < 250 or page_image_names):
        pix = page.get_pixmap(dpi=IMAGE_DPI, alpha=False)
        render_name = f"page-{page_num:04d}.png"
        pix.save(PAGE_RENDERS_DIR / render_name)

    record = {
        "page": page_num,
        "text": text,
        "ocr_used": ocr_used,
        "embedded_images": [f"figures/{n}" for n in page_image_names],
        "page_render": f"page_renders/{render_name}" if render_name else None
    }
    page_records.append(record)

    page_md = [f"\n\n## PDF page {page_num}\n"]
    page_md.append(text if text else "_No text could be extracted from this page._")
    if page_image_names:
        page_md.append("\n\n**Embedded images:**")
        page_md.extend([f"\n- `figures/{name}`" for name in page_image_names])
    if render_name:
        page_md.append(f"\n\n_Page render: `page_renders/{render_name}`_")
    markdown_parts.append("".join(page_md))

pdf_metadata = pdf.metadata or {}
print(f"Pages processed: {len(page_records)}")
print(f"Embedded images extracted: {len(image_records)}")
print(f"Pages OCR-processed: {sum(r['ocr_used'] for r in page_records)}")
print(f"PDF metadata: {pdf_metadata}")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Pages processed: 272
Embedded images extracted: 84
Pages OCR-processed: 10
PDF metadata: {'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'producer': 'Adobe PDF Library 18.0', 'creationDate': "D:20260625160525-04'00'", 'modDate': "D:20260625160659-04'00'", 'trapped': '', 'encryption': None}


## 4. Create Markdown, page records, and searchable chunks

Chunks are split on word boundaries, with overlap so a concept spanning a boundary is less likely to be lost. Each chunk includes the page or pages it came from. The original page-level file remains the authoritative reference for quotations and page citations.

In [ ]:
# Main page-labelled Markdown
title = PDF_PATH.stem.replace("_", " ")
doc_md = [
    f"# {title}",
    "",
    f"Source PDF: `{PDF_PATH.name}`",
    f"Total PDF pages: {len(page_records)}",
    "",
    "> Generated text extraction. Check the original PDF for exact typography, layout, and visual details.",
    ""
]
doc_md.extend(markdown_parts)
(OUTPUT_DIR / "dissertation.md").write_text("\n".join(doc_md), encoding="utf-8")

# Page-level JSONL
with (OUTPUT_DIR / "pages.jsonl").open("w", encoding="utf-8") as f:
    for rec in page_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Image inventory
with (OUTPUT_DIR / "figures.jsonl").open("w", encoding="utf-8") as f:
    for rec in image_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Word-based chunks, retaining source page ranges.
def words_with_pages(records):
    all_words = []
    for rec in records:
        for w in rec["text"].split():
            all_words.append((w, rec["page"]))
    return all_words

word_items = words_with_pages(page_records)
chunks = []
step = max(1, CHUNK_WORDS - CHUNK_OVERLAP_WORDS)

for start in range(0, len(word_items), step):
    part = word_items[start:start + CHUNK_WORDS]
    if not part:
        break
    pages = sorted(set(p for _, p in part))
    chunks.append({
        "chunk_id": f"chunk-{len(chunks)+1:05d}",
        "start_page": min(pages),
        "end_page": max(pages),
        "pages": pages,
        "text": " ".join(w for w, _ in part)
    })
    if start + CHUNK_WORDS >= len(word_items):
        break

with (OUTPUT_DIR / "chunks.jsonl").open("w", encoding="utf-8") as f:
    for rec in chunks:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Created {len(chunks)} chunks.")
print(f"Markdown size: {(OUTPUT_DIR / 'dissertation.md').stat().st_size:,} bytes")

Created 116 chunks.
Markdown size: 543,519 bytes


## 5. Optional: run Docling for layout-aware Markdown

Docling can preserve more document structure than plain text extraction. Its result is saved separately so you can compare it with the page-labelled PyMuPDF extraction. This step can take several minutes and may use substantial Colab memory. If it fails, the other outputs remain available.

In [ ]:
docling_status = "skipped"
if RUN_DOCLING:
    try:
        from docling.document_converter import DocumentConverter
        converter = DocumentConverter()
        conversion = converter.convert(str(PDF_PATH))
        docling_markdown = conversion.document.export_to_markdown()
        (OUTPUT_DIR / "docling.md").write_text(docling_markdown, encoding="utf-8")
        docling_status = "success"
        print(f"Docling Markdown saved ({len(docling_markdown):,} characters).")
    except Exception as e:
        docling_status = f"failed: {type(e).__name__}: {e}"
        print("Docling did not complete. The page-level extraction is still available.")
        print(docling_status)
else:
    print("Docling disabled in setup cell.")

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

ERROR:docling.pipeline.standard_pdf_pipeline:Stage ocr failed for run 1: Stage ocr stopped during its model call
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/docling/pipeline/standard_pdf_pipeline.py", line 392, in _process_batch
    processed_pages = self._run_model(good[0].conv_res, pages)
  File "/usr/local/lib/python3.13/dist-packages/docling/pipeline/standard_pdf_pipeline.py", line 287, in _run_model
    raise RuntimeError(f"Stage {self.name} stopped during its model call")
RuntimeError: Stage ocr stopped during its model call
ERROR:docling.pipeline.standard_pdf_pipeline:Stage layout failed for run 1: Stage layout stopped during its model call
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/docling/pipeline/standard_pdf_pipeline.py", line 392, in _process_batch
    processed_pages = self._run_model(good[0].conv_res, pages)
  File "/usr/local/lib/python3.13/dist-packages/docling/pipeline/standard_pdf_pipeline.

KeyboardInterrupt: 

## 6. Write a manifest and package the results

The ZIP includes the extracted research materials, not the original PDF. Keep the source PDF in Drive as the authoritative original. You can copy the ZIP contents into your OpenCode project or point OpenCode at the extracted folder.

In [ ]:
manifest = {
    "source_pdf_name": PDF_PATH.name,
    "source_pdf_size_bytes": PDF_PATH.stat().st_size,
    "pdf_page_count": len(page_records),
    "embedded_image_count": len(image_records),
    "ocr_pages": [r["page"] for r in page_records if r["ocr_used"]],
    "chunk_count": len(chunks),
    "chunk_words": CHUNK_WORDS,
    "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
    "docling_status": docling_status,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notes": [
        "Page numbers refer to PDF page positions, starting at 1.",
        "Extracted text may differ from visual reading order in complex layouts.",
        "Check the original PDF for exact quotations, diagrams, and page layout.",
        "Some vector graphics may only be present in page renders or the original PDF."
    ]
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

import shutil
zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Research package:", OUTPUT_DIR)
print("ZIP archive:", zip_path)
print(f"ZIP size: {Path(zip_path).stat().st_size / (1024*1024):.1f} MB")

## 7. Download the ZIP (optional)

The package is already saved in Google Drive beside the source PDF. Use this cell if you also want to download a copy to your computer.

In [ ]:
from google.colab import files
files.download(zip_path)

## 8. Suggested OpenCode project instructions

Copy this into the `AGENTS.md` file at the root of the OpenCode research project, and adjust the source folder name if needed.

```markdown
# Dissertation research source

The `knowledge/` folder contains a processed research copy of the dissertation.

## Research rules
- Treat the original PDF as authoritative.
- Use `knowledge/pages.jsonl` and `knowledge/dissertation.md` to locate relevant passages.
- Cite PDF page numbers in every substantive answer about the dissertation.
- When using `chunks.jsonl`, inspect the corresponding page records before quoting.
- Distinguish the author's claims from your own interpretation.
- Do not invent quotations, page numbers, figure captions, or bibliographic details.
- Refer to extracted images by their filenames and original PDF page numbers.
- For visual interpretation, inspect the corresponding image in `knowledge/figures/`
  or `knowledge/page_renders/`; do not rely only on OCR or generated descriptions.
- Preserve the original source and do not silently rewrite extracted text.
```

## Next step: semantic search

Once this basic package works, you can add embeddings and a local vector index (for example, ChromaDB or FAISS). That will let OpenCode retrieve conceptually relevant passages from the dissertation. It is better to validate the extracted text, page references, and figures first.